# Supermercado

In [ ]:
# Importamos las librerías necesarias
import sqlite3  # Permite trabajar con bases de datos SQLite
import pandas as pd  # Para mostrar resultados en forma de tabla

In [ ]:
# Creamos la conexión a una base de datos temporal en memoria

# ':memory:' crea una base de datos que solo vive mientras el programa está activo.
# Si quisiéramos guardar la base de datos en un archivo, podríamos usar 'supermercado.db'
conn = sqlite3.connect(':memory:')

# El "cursor" es el objeto que nos permite ejecutar sentencias SQL en la base de datos
cursor = conn.cursor()


## Creación tablas

In [ ]:
# Creamos la tabla productos

# TEXT: string
# INTEGER: int
# REAL: decimal
cursor.execute('''
CREATE TABLE IF NOT EXISTS productos(
    id_producto INTEGER PRIMARY KEY NOT NULL,
    nombre TEXT,
    descripcion TEXT,
    precio REAL,
    stock INTEGER
);
''')


In [ ]:
# Creamos la tabla pedidos

# Crear una tabla llamada “pedidos” con sus 4 columnas denominadas
# “id_pedido”, “id_producto”, “cantidad”, “fecha” que sean de
# tipo “entero, entero, entero, fecha”,
# teniendo en cuenta que la PK es el campo “id_pedido”.
# Además, hacer referencia a la tabla “productos” usando el campo “id_producto” como FK.

# TEXT: string, date
# INTEGER: int
# REAL: decimal

cursor.execute('''
CREATE TABLE IF NOT EXISTS pedidos(
    id_pedido INTEGER UNSIGNED PRIMARY KEY NOT NULL,
    id_producto INTEGER,
    cantidad INTEGER UNSIGNED,
    fecha TEXT,
    FOREIGN KEY (id_producto) REFERENCES productos (id_producto)
);
''')


In [ ]:
# Lista de productos
productos = [
    (1, 'Leche', 'Leche entera en envase de 1L', 1.20, 100),
    (2, 'Queso', 'Queso curado en bloque de 500g', 4.50, 50),
    (3, 'Yogurt', 'Yogurt natural sin azúcar en envase de 500g', 2.00, 80),
    (4, 'Pan', 'Pan blanco recién horneado en barra de 500g', 1.50, 200),
    (5, 'Huevos', 'Huevos de gallina campera de tamaño L en docena', 2.80, 30),
    (6, 'Arroz', 'Arroz blanco de grano largo en paquete de 1Kg', 1.70, 150),
    (7, 'Pasta', 'Pasta italiana en paquete de 500g', 2.30, 120),
    (8, 'Carne', 'Carne de ternera en bandeja de 500g', 8.90, 20),
    (9, 'Pescado', 'Pescado fresco de la costa en pieza de 1Kg', 12.50, 10),
    (10, 'Frutas', 'Cesta de frutas variadas de temporada', 7.20, 15),
    (11, 'Mantequilla', 'Mantequilla sin sal 250g', 1.80, 15),
    (12, 'Cereales', 'Caja de cereales para el desayuno', 2.50, 50)
]

# Insertamos los productos
cursor.executemany('''
    INSERT INTO productos (id_producto, nombre, descripcion, precio, stock)
    VALUES (?, ?, ?, ?, ?)
''', productos)

In [ ]:
# Insertamos pedidos en la tabla pedidos
pedidos = [
    (1, 3, 2, '2023-04-15'),
    (2, 5, 1, '2023-04-16'),
    (3, 1, 4, '2023-04-16'),
    (4, 8, 3, '2023-04-16'),
    (5, 2, 2, '2023-04-17'),
    (6, 9, 1, '2023-04-17'),
    (7, 4, 3, '2023-04-17'),
    (8, 8, 1, '2023-04-17'),
    (9, 6, 2, '2023-04-17'),
    (10, 10, 2, '2023-04-18'),
    (11, 8, 5, '2023-04-18'),
    (12, 7, 4, '2023-04-18')
]

cursor.executemany('''
INSERT INTO pedidos (id_pedido, id_producto, cantidad, fecha)
VALUES (?, ?, ?, ?)
''', pedidos)


In [ ]:
# guardar los cambios en la bbdd
conn.commit()

## Queries básicas

In [ ]:
# 1. Seleccionar todos los productos
pd.read_sql_query('''
SELECT * FROM productos;
''', conn)


,id_producto,nombre,descripcion,precio,stock
0,1,Leche,Leche entera en envase de 1L,1.2,100
1,2,Queso,Queso curado en bloque de 500g,4.5,50
2,3,Yogurt,Yogurt natural sin azúcar en envase de 500g,2.0,80
3,4,Pan,Pan blanco recién horneado en barra de 500g,1.5,200
4,5,Huevos,Huevos de gallina campera de tamaño L en docena,2.8,30
5,6,Arroz,Arroz blanco de grano largo en paquete de 1Kg,1.7,150
6,7,Pasta,Pasta italiana en paquete de 500g,2.3,120
7,8,Carne,Carne de ternera en bandeja de 500g,8.9,20
8,9,Pescado,Pescado fresco de la costa en pieza de 1Kg,12.5,10
9,10,Frutas,Cesta de frutas variadas de temporada,7.2,15


In [ ]:
# 1b. Seleccionar todos los pedidos
pd.read_sql_query('''
SELECT * FROM pedidos;
''', conn)

,id_pedido,id_producto,cantidad,fecha
0,1,3,2,2023-04-15
1,2,5,1,2023-04-16
2,3,1,4,2023-04-16
3,4,8,3,2023-04-16
4,5,2,2,2023-04-17
5,6,9,1,2023-04-17
6,7,4,3,2023-04-17
7,8,8,1,2023-04-17
8,9,6,2,2023-04-17
9,10,10,2,2023-04-18


In [ ]:
# 2. Seleccionar los productos con un stock menor o igual a 10
pd.read_sql_query('''
SELECT * FROM productos
WHERE stock <= 10;
''', conn)

,id_producto,nombre,descripcion,precio,stock
0,9,Pescado,Pescado fresco de la costa en pieza de 1Kg,12.5,10


In [ ]:
# 3. Seleccionar el nombre y precio del producto con el mayor precio:
pd.read_sql_query('''
SELECT nombre, precio FROM productos
WHERE precio = (SELECT MAX(precio) FROM productos);
''', conn)

,nombre,precio
0,Pescado,12.5


In [ ]:
# Esta opcion no me serviria porque hay que ordenar la tabla,
#y si hay empates habria que ajustar los limites
#aparte de todo el tema de rendimiento
pd.read_sql_query('''
SELECT nombre, precio  FROM productos
ORDER BY precio DESC
LIMIT 1;
''', conn)

,nombre,precio
0,Pescado,12.5


In [ ]:
# 4. Seleccionar todos los productos con un precio mayor a 5 euros:
pd.read_sql_query('''
SELECT * FROM productos
WHERE precio > 5;
''', conn)

,id_producto,nombre,descripcion,precio,stock
0,8,Carne,Carne de ternera en bandeja de 500g,8.9,20
1,9,Pescado,Pescado fresco de la costa en pieza de 1Kg,12.5,10
2,10,Frutas,Cesta de frutas variadas de temporada,7.2,15


In [ ]:
# 5. Seleccionar el nombre y descripción de los productos
# que contienen la palabra "ternera" en su descripción:
pd.read_sql_query('''
SELECT nombre, descripcion FROM productos
WHERE descripcion LIKE '%ternera%'
''', conn)

,nombre,descripcion
0,Carne,Carne de ternera en bandeja de 500g


In [ ]:
# 6. Mostrar los nombres únicos de todos los productos disponibles en la tienda:
pd.read_sql_query('''
SELECT DISTINCT nombre FROM productos;
''', conn)

,nombre
0,Leche
1,Queso
2,Yogurt
3,Pan
4,Huevos
5,Arroz
6,Pasta
7,Carne
8,Pescado
9,Frutas


In [ ]:
# 7. Seleccionar el id_pedido y la cantidad de todos los pedidos:
pd.read_sql_query('''
SELECT id_pedido, cantidad FROM pedidos;
''', conn)

,id_pedido,cantidad
0,1,2
1,2,1
2,3,4
3,4,3
4,5,2
5,6,1
6,7,3
7,8,1
8,9,2
9,10,2


In [ ]:
# 8. Seleccionar todos los pedidos que se realizaron en el día 2023-04-17:
pd.read_sql_query('''
SELECT * FROM pedidos
WHERE fecha = '2023-04-17';
''', conn)

,id_pedido,id_producto,cantidad,fecha
0,5,2,2,2023-04-17
1,6,9,1,2023-04-17
2,7,4,3,2023-04-17
3,8,8,1,2023-04-17
4,9,6,2,2023-04-17


In [ ]:
# 9. Seleccionar todos los pedidos que se realizaron en abril de 2023:
pd.read_sql_query('''
SELECT * FROM pedidos
WHERE fecha >= '2023-04-01' AND fecha < '2023-05-01';
''', conn)

,id_pedido,id_producto,cantidad,fecha
0,1,3,2,2023-04-15
1,2,5,1,2023-04-16
2,3,1,4,2023-04-16
3,4,8,3,2023-04-16
4,5,2,2,2023-04-17
5,6,9,1,2023-04-17
6,7,4,3,2023-04-17
7,8,8,1,2023-04-17
8,9,6,2,2023-04-17
9,10,10,2,2023-04-18


In [ ]:
pd.read_sql_query('''
SELECT * FROM pedidos
WHERE fecha LIKE '2023-04-%';
''', conn)

,id_pedido,id_producto,cantidad,fecha
0,1,3,2,2023-04-15
1,2,5,1,2023-04-16
2,3,1,4,2023-04-16
3,4,8,3,2023-04-16
4,5,2,2,2023-04-17
5,6,9,1,2023-04-17
6,7,4,3,2023-04-17
7,8,8,1,2023-04-17
8,9,6,2,2023-04-17
9,10,10,2,2023-04-18


In [ ]:
# 10. Seleccionar la cantidad total del producto número 8 en todos los pedidos:
pd.read_sql_query('''
SELECT id_producto, SUM(cantidad) AS total_cantidad
FROM pedidos
WHERE id_producto = 8;
''', conn)

,id_producto,total_cantidad
0,8,9


In [ ]:
# 11. Seleccionar los pedidos que contengan productos con id_producto 1, 3 o 5:
pd.read_sql_query('''
SELECT * FROM pedidos
WHERE id_producto IN (1, 3, 5);
''', conn)


,id_pedido,id_producto,cantidad,fecha
0,1,3,2,2023-04-15
1,2,5,1,2023-04-16
2,3,1,4,2023-04-16



### JOINS


In [ ]:
# Seleccionar los pedidos que se hayan realizado
# en una fecha específica (ayer: 2023-04-17)
# y los nombres de los productos correspondientes:
pd.read_sql_query('''

''', conn)


In [ ]:
# Seleccionar los productos
# que tienen un stock menor o igual a 10 unidades
# y los pedidos correspondientes,
# incluyendo aquellos productos que no tienen ningún pedido:
pd.read_sql_query('''

''', conn)

## Otros

In [ ]:
df_productos = pd.read_csv("productos.csv")
df_pedidos = pd.read_csv("pedidos.csv")

In [ ]:
# Insertar productos en la tabla ya creada
df_productos.to_sql('productos', conn, if_exists='append', index=False)

# Insertar pedidos en la tabla ya creada
df_pedidos.to_sql('pedidos', conn, if_exists='append', index=False)


In [ ]:
pd.read_sql_query("SELECT * FROM productos", conn)
